# EcoHome Energy Advisor - RAG Setup

In this notebook, we'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Retrieval quality is scored with Ragas in `03_run_and_evaluate.ipynb`


## 1. Import Required Libraries


In [1]:
# Import the necessary libraries for RAG setup
import os
import glob
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv


In [2]:
load_dotenv()


True

## 2. Load and Process Documents


In [5]:
# Load the energy-saving tip documents
# Load both starter tips and the extra topic documents in data/documents/
# Use TextLoader to load the documents

documents = []
document_paths = sorted(glob.glob("data/documents/*.txt"))
if not document_paths:
    print("No text files found in data/documents/")
else:
    for doc_path in document_paths:
        loader = TextLoader(doc_path)
        docs = loader.load()
        documents.extend(docs)
        print(f"Loaded {len(docs)} documents from {doc_path}")

print(f"Total documents loaded: {len(documents)}")


Loaded 1 documents from data/documents/tip_device_best_practices.txt
Loaded 1 documents from data/documents/tip_energy_savings.txt
Loaded 1 documents from data/documents/tip_energy_storage_optimization.txt
Loaded 1 documents from data/documents/tip_hvac_optimization_strategies.txt
Loaded 1 documents from data/documents/tip_renewable_energy_integration.txt
Loaded 1 documents from data/documents/tip_seasonal_energy_management.txt
Loaded 1 documents from data/documents/tip_smart_home_automation.txt
Total documents loaded: 7


## 3. Split Documents into Chunks


In [4]:
# Split documents into smaller chunks for better retrieval
# Use RecursiveCharacterTextSplitter with appropriate chunk_size and chunk_overlap
# Experiment with different chunk sizes (e.g., 500, 1000, 1500 characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 7 documents into 17 chunks

Sample chunk (first 200 characters):
Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during...


## 4. Create Vector Store


In [6]:
# Create a ChromaDB vector store
# Initialize OpenAIEmbeddings
# Create the vector store with the document chunks
# Persist the vector store to disk for future use

# Set up the persist directory
persist_directory = "data/vectorstore"
os.makedirs(persist_directory, exist_ok=True)

# Initialize embeddings
embedding_kwargs = {
    "model": os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
}
base_url = os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE")
if base_url:
    embedding_kwargs["base_url"] = base_url
embeddings = OpenAIEmbeddings(**embedding_kwargs)

# Create the vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector store created and persisted to {persist_directory}")
print(f"Total vectors stored: {len(splits)}")


Vector store created and persisted to data/vectorstore
Total vectors stored: 17


## 5. Test the RAG Pipeline


In [7]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling",
    "home battery storage",
    "seasonal energy management",
    "smart home automation",
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    docs = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(docs):
        print(f"  Result {i+1}: {doc.page_content[:100]}...")


=== Testing Vector Search ===

Query: 'electric vehicle charging tips'
  Result 1: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
  Result 2: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...

Query: 'thermostat optimization'
  Result 1: HVAC Optimization Strategies

Heating and cooling often account for 40-50% of home electricity use. ...
  Result 2: HVAC Optimization Strategies

Heating and cooling often account for 40-50% of home electricity use. ...

Query: 'dishwasher energy saving'
  Result 1: Appliance and EV automations:
- Put dishwashers, washers, dryers, and EV chargers on delayed start. ...
  Result 2: Appliance and EV automations:
- Put dishwashers, washers, dryers, and EV chargers on delayed start. ...

Query: 'solar power maximization'
  Result 1: Inverters, export, and interconnection:
- Many utilities pay less for exported solar than they charg...
  Result 2: I

## 6. Test the Search Tool


In [7]:
# Test the search_energy_tips tool from tools.py
# Import and test the tool with various queries
# Verify that it returns relevant results

from tools import search_energy_tips

# Test the search_energy_tips function
print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips",
    "energy storage optimization",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    result = search_energy_tips.invoke(
        input={
            "query": query,
            "max_results": 3,
        }
    )

    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for i, tip in enumerate(result['tips']):
            print(f"    {i+1}. {tip['content'][:100]}...")
            print(f"       Source: {tip['source']}")
            print(f"       Relevance: {tip['relevance_score']}")


=== Testing search_energy_tips Tool ===

Query: 'electric vehicle charging'


  Found 3 results
    1. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: data/documents/tip_device_best_practices.txt
       Relevance: high
    2. Energy Storage Optimization

A home battery is most valuable when it stores cheap or solar energy an...
       Source: data/documents/tip_energy_storage_optimization.txt
       Relevance: high
    3. Control rules that work well:
- If solar irradiance is high and prices are mid-peak, run flexible lo...
       Source: data/documents/tip_smart_home_automation.txt
       Relevance: medium

Query: 'thermostat settings'


  Found 3 results
    1. HVAC Optimization Strategies

Heating and cooling often account for 40-50% of home electricity use. ...
       Source: data/documents/tip_hvac_optimization_strategies.txt
       Relevance: high
    2. Equipment and airflow:
- Replace or clean HVAC filters every 30-90 days. A dirty filter makes the bl...
       Source: data/documents/tip_hvac_optimization_strategies.txt
       Relevance: high
    3. Spring and fall:
- These are the best seasons for deep cleaning filters, coils, and dryer vents.
- U...
       Source: data/documents/tip_seasonal_energy_management.txt
       Relevance: medium

Query: 'dishwasher optimization'


  Found 3 results
    1. Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...
       Source: data/documents/tip_device_best_practices.txt
       Relevance: high
    2. Appliance and EV automations:
- Put dishwashers, washers, dryers, and EV chargers on delayed start. ...
       Source: data/documents/tip_smart_home_automation.txt
       Relevance: high
    3. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: data/documents/tip_device_best_practices.txt
       Relevance: medium

Query: 'solar power tips'


  Found 3 results
    1. Inverters, export, and interconnection:
- Many utilities pay less for exported solar than they charg...
       Source: data/documents/tip_renewable_energy_integration.txt
       Relevance: high
    2. Renewable Energy Integration

Homes with rooftop solar save the most when flexible loads follow the ...
       Source: data/documents/tip_renewable_energy_integration.txt
       Relevance: high
    3. Sizing the opportunity:
- Shifting 10 kWh from on-peak ($0.33/kWh) to off-peak ($0.13/kWh) saves abo...
       Source: data/documents/tip_energy_storage_optimization.txt
       Relevance: medium

Query: 'energy storage optimization'


  Found 3 results
    1. Energy Storage Optimization

A home battery is most valuable when it stores cheap or solar energy an...
       Source: data/documents/tip_energy_storage_optimization.txt
       Relevance: high
    2. Sizing the opportunity:
- Shifting 10 kWh from on-peak ($0.33/kWh) to off-peak ($0.13/kWh) saves abo...
       Source: data/documents/tip_energy_storage_optimization.txt
       Relevance: high
    3. Seasonal Energy Management

Energy use changes with the season. The same device schedule should not ...
       Source: data/documents/tip_seasonal_energy_management.txt
       Relevance: medium
